# 09 · ¿Memoriza? Las cuatro figuras del argumento

La val loss tiene mínimo cerca de los 30k y después sube. Este notebook contesta **dónde** ocurre eso, **hacia qué función**
converge la red, y si las muestras finales son **copias** del training set. Cuatro figuras, cada una con una pregunta:

| fig | pregunta | respuesta que da |
|---|---|---|
| **1** | ¿dónde vive la subida de la val loss? | localizada en $t$: deciles bajos y medios |
| **2** | ¿hacia qué función converge la red? | al score empírico del **train** (memorización, medida) |
| **3** | ¿el modelo con peor val loss genera peor? | no — la disociación |
| **4** | ¿las muestras son copias del train? | la distribución de distancias al NN, contra su nulo |

## El objeto de la Figura 2, que es el corazón del trabajo

Para un dataset **finito** $\{x_0^{(i)}\}_{i=1}^N$, la marginal del VP-SDE es una mixtura gaussiana **exacta**:

$$p_t(x)=\frac1N\sum_i \mathcal N\!\left(x;\;\bar\alpha_t x_0^{(i)},\;\sigma_t^2 I\right)$$

y por Tweedie su score sale en forma cerrada vía la media posterior de $x_0$:

$$\hat x_0^{\text{emp}}(x,t)=\sum_i w_i(x,t)\,x_0^{(i)},\qquad
w_i\propto\exp\!\left(-\frac{\|x-\bar\alpha_t x_0^{(i)}\|^2}{2\sigma_t^2}\right),\qquad
\nabla\log p_t(x)=-\frac{x-\bar\alpha_t\hat x_0^{\text{emp}}}{\sigma_t^2}$$

El teorema de DSM dice que el minimizador del objetivo **sobre todas las funciones medibles** es ese score. O sea: está escrito
el punto al que el entrenamiento tiende. Medimos la distancia a ese punto,

$$E(t)=\frac{\mathbb E_x\left\|\hat x_0^{\theta}(x,t)-\hat x_0^{\text{emp}}(x,t)\right\|^2}{\mathbb E_x\left\|\hat x_0^{\text{emp}}(x,t)\right\|^2}$$

con $\hat x_0^{\theta}=(x+\sigma_t^2 s_\theta)/\bar\alpha_t$, en dos versiones: $\hat x_0^{\text{emp}}$ construido con el
**train** y con el **held-out**. Ninguna métrica perceptual, ninguna red preentrenada.

> **La identidad de Tweedie está verificada numéricamente en el notebook** (celda de la Figura 2) contra el gradiente por
> autograd del `logsumexp`, a ~$10^{-8}$ relativo. No es una fórmula copiada: es una implementación chequeada.

## Tres honestidades que van en el texto de la monografía

1. **El nivel del panel held-out no significa nada, solo la forma.** El score empírico del held-out **no** es el score
   poblacional: es otra aproximación finita y ruidosa. Descomponiendo, la distancia al held-out ≈ distancia al poblacional +
   un offset que **no depende del checkpoint**. Entonces el mínimo cae en el lugar correcto pero el valor absoluto no es
   interpretable.
2. **La Figura 4 es el caso límite $t\to0$ de la Figura 2.** Cuando $\sigma_t\to0$ los pesos $w_i$ colapsan a one-hot y
   $\hat x_0^{\text{emp}}$ degenera en "el vecino más cercano". El chequeo de NN no es un apéndice: es la misma medición en el
   extremo. El notebook **mide** $\max_i w_i$ por decil, así que se ve exactamente en qué régimen está cada bin — y en
   dimensión $3\cdot64\cdot64=12288$ el colapso empieza bastante antes de $t\to0$.
3. **La Figura 3 no puede probar nada sobre diversidad ni calidad promedio.** Con 4 muestras es evidencia de que no hay
   degradación visible, que es todo lo que el argumento necesita. No se sobre-vende.

## Y una corrección al plan

La "trampa" de la Figura 1 —que el offset de nivel entre train y val sea arbitrario por sortear $t$ y $\varepsilon$ de forma
independiente— **ya está resuelta en el código**. En `training/trainer.py` los dos exámenes fijos se construyen con el
**mismo objeto `time_sampler`** y la **misma `VAL_EXAM_SEED`**, y el comentario del fuente lo dice explícito: *"ambos exámenes
sortean la misma secuencia de t y de ruido, de modo que la diferencia entre las dos curvas no pueda venir del examen"*. Como
los dos conjuntos tienen **el mismo tamaño** (465 y 465) y el mismo `batch_size`, los $t_i$ y $\varepsilon_i$ salen idénticos
índice por índice: **los niveles ya son comparables**, no solo las pendientes.

Igual el examen se **reconstruye acá**, por dos razones que no son la del plan: (a) el `.jsonl` guarda solo escalares, así que
la loss por muestra y por decil **no existe** y hay que recomputarla; (b) conviene un **grid estratificado** de $t$ —un $t$ por
(imagen, decil), sorteado dentro del bin— en vez de un $t$ por imagen, porque garantiza el mismo $n$ en los diez bins. La
celda de abajo **verifica** que los dos conjuntos tengan igual tamaño en vez de asumirlo: si algún día difieren, la alineación
se rompe y la advertencia del plan vuelve a valer.

## Artefactos y costo

Las figuras 2 y 4 son las caras. Todo lo pesado se **cachea** en `data/_cache_09/`, así que las figuras se re-dibujan sin
recomputar; `RECALCULAR = True` fuerza todo de nuevo. Los **latentes fijos** se generan una vez con semilla propia y se
guardan a disco: es lo que hace que el caption *"los latentes se fijaron a priori y se reusan en todas las figuras"* sea
auditable y no una promesa.

In [ ]:
# --- Setup ---
import sys
import pathlib
import os
import re
import json
import time
from collections import namedtuple

_here = pathlib.Path.cwd()
_root = None
for _cand in (_here, *_here.parents):
    if (_cand / "src" / "diffusion").is_dir():
        _root = _cand
        break
if _root is None:
    raise RuntimeError(f"No encontré src/diffusion subiendo desde {_here}")
_src = str((_root / "src").resolve())
if _src not in sys.path:
    sys.path.insert(0, _src)

import numpy as np
import torch
import matplotlib.pyplot as plt
import yaml
from scipy.stats import ks_2samp

from diffusion.data_generation import count_images, finite_batches
from diffusion.models import EpsilonScoreWrapper, make_model
from diffusion.samplers import available_time_grids, make_sampler
from diffusion.sde import make_sde
from diffusion.training import load_checkpoint

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
plt.rcParams.update({"figure.dpi": 130, "axes.grid": True, "grid.alpha": 0.25})

# --- NFE, no pasos (misma convención que 07/08, más el denoise final) -----------
# Evaluaciones del score por PASO: euler y pf_ode 1, heun 2 (drift en t y en el
# predicho a t+dt), pc 1+n_corrector. Y ADEMÁS: sample() cierra con un paso de
# denoising de Tweedie (`denoise=True` por default) que cuesta 1 evaluación extra,
# igual para los cuatro. O sea NFE = pasos·(por paso) + 1. La celda siguiente lo
# verifica contando llamadas reales: el conteo no se asume, se mide.
N_CORRECTOR = 1
NFE_POR_PASO = {"euler": 1, "pf_ode": 1, "heun": 2, "pc": 1 + N_CORRECTOR}


def pasos_para(nom, nfe):
    """Pasos que caben en un presupuesto de `nfe` evaluaciones (denoise final incluido)."""
    return max(1, (nfe - 1) // NFE_POR_PASO[nom])


def nfe_real(nom, pasos):
    """NFE efectivamente gastadas. Para heun/pc solo son alcanzables valores impares."""
    return pasos * NFE_POR_PASO[nom] + 1


def kwargs_para(nom):
    return {"n_corrector": N_CORRECTOR} if nom == "pc" else {}


# --- Perillas -------------------------------------------------------------------
RECALCULAR = False          # True fuerza recomputar todo lo cacheado

N_DECILES = 10
ESCALA_T = "log"            # "log" (deciles log-uniformes, como time_sampling) o "lineal"
SEED_EXAMEN = 20_260_804    # t y eps del examen de la Figura 1 (compartidos train/val)

N_EVAL_E = 256              # puntos x donde se estima la esperanza de E(t)
SEED_EVAL_E = 777

SAMPLER_FIG = "heun"        # sampler de las figuras 3 y 4
NFE_FIG = 64                # presupuesto de evaluaciones (el denoise final cuenta)
GRILLA_T = "uniform"        # grilla temporal del sampler: ver available_time_grids().
                            # "uniform" es el default de la librería y lo que usaron 06/07/08.
                            # A 64 NFE la grilla PESA: "logsnr" reparte los pasos donde el
                            # score cambia rápido y suele ganarle a uniform con pocos pasos.
                            # Lo que importa para estas figuras es que sea LA MISMA en los
                            # tres checkpoints, y lo es porque es una constante global.
N_METRICAS = 1000           # latentes fijos reusados por TODAS las figuras
SEED_LATENTES = 12_345
PASOS_FIG3 = [15_000, 30_000, 120_000]
PASOS_FIG4 = [30_000, 120_000]
N_VECINOS = 3               # vecinos a mostrar en el panel A de la Figura 4
LOTE_GEN = 50               # batch de generación (bajalo si falta VRAM)
LOTE_DIST = 256             # filas por bloque en las matrices de distancia

CACHE = _root / "data" / "_cache_09"
CACHE.mkdir(parents=True, exist_ok=True)


def cacheado(nombre, fn):
    """Ejecuta fn() y cachea su dict de arrays en un .npz; lo relee si ya está."""
    p = CACHE / f"{nombre}.npz"
    if p.exists() and not RECALCULAR:
        with np.load(p, allow_pickle=False) as z:
            print(f"  [cache] {nombre} <- {p.name}")
            return {k: z[k] for k in z.files}
    t0 = time.time()
    d = fn()
    np.savez_compressed(p, **d)
    print(f"  [calc ] {nombre} en {(time.time()-t0)/60:.1f} min -> {p.name}")
    return d


def denorm(t):
    """(B,3,H,W) en [-1,1] -> (B,H,W,3) numpy en [0,1]."""
    return (t.detach() * 0.5 + 0.5).clamp(0, 1).permute(0, 2, 3, 1).cpu().numpy()


PASOS_FIG = pasos_para(SAMPLER_FIG, NFE_FIG)
NFE_EFECTIVO = nfe_real(SAMPLER_FIG, PASOS_FIG)

print("paquete en:", _root)
print("torch:", torch.__version__, "| device:", DEVICE)
print(f"grillas temporales disponibles: {available_time_grids()} | usando '{GRILLA_T}'")
print(f"{SAMPLER_FIG}: presupuesto {NFE_FIG} NFE -> {PASOS_FIG} pasos "
      f"= {NFE_EFECTIVO} NFE efectivas ({PASOS_FIG}x{NFE_POR_PASO[SAMPLER_FIG]} + 1 del "
      "denoise final)")
print("cache:", CACHE)
if DEVICE == "cpu":
    print()
    print("AVISO: sin GPU. Las figuras 2 y 4 son intensivas (score empírico contra todo el")
    print("train, y 2x1000 muestras). En CPU no cierra: corré esto en la máquina del lab.")

In [ ]:
# --- El conteo de NFE, medido contra el código instalado (no asumido) ---
# Se envuelve score_fn en un contador y se samplea con una SDE 2D barata. Si alguien
# cambia el driver —p. ej. agregando o quitando el denoise final— esta celda lo detecta
# acá, en vez de dejar mal rotuladas todas las figuras.
_sde2 = make_sde("vp", data_dim=2)
_P = 7
print(f"{'sampler':8s} {'NFE/paso':>9s} {'esperado':>9s} {'medido':>7s} {'ok':>4s}"
      f"   (n_steps={_P}, grilla '{GRILLA_T}')")
_todo_ok = True
for _nom in ("euler", "pf_ode", "heun", "pc"):
    _cnt = [0]

    def _sc(x, t, _c=_cnt):
        _c[0] += 1
        return torch.zeros_like(x)

    _smp = make_sampler(_nom, _sde2, _sc, n_steps=_P, t_eps=1e-3, time_grid=GRILLA_T,
                        **kwargs_para(_nom))
    _smp.sample(3, generator=torch.Generator().manual_seed(0))
    _esp = nfe_real(_nom, _P)
    _ok = _esp == _cnt[0]
    _todo_ok &= _ok
    print(f"{_nom:8s} {NFE_POR_PASO[_nom]:9d} {_esp:9d} {_cnt[0]:7d} "
          f"{'OK' if _ok else 'MAL':>4s}")
if not _todo_ok:
    raise AssertionError(
        "El conteo de NFE no coincide con el código instalado: el driver de sample() cambió. "
        "Ajustá NFE_POR_PASO / nfe_real antes de interpretar cualquier figura."
    )
print("\ntabla de NFE verificada contra el código")

# El paso final de denoising del sampler ES x̂0^θ: E[x_0|x_t] = (x + σ² s)/α. O sea que
# el objeto central de la Figura 2 es, literalmente, la última operación que hace el
# sampler. La equivalencia se chequea más abajo, cuando x0_theta ya está definida.

## 0. Las fuentes: checkpoints y datasets

Los checkpoints salen de las **dos** corridas (`cats_vp.yaml` con 60k pasos y `cats_vp-overfitting.yaml` con 120k), que son la
misma configuración salvo el horizonte y el directorio de salida. Con `keep_last_checkpoints: 4`, los snapshots de 15k/30k/45k
**solo sobreviven en `base`** y los de 75k…120k solo en `overfit`; el de 60k está en las dos.

Eso importa para las figuras 3 y 4: **15k y 30k vienen de `base`, 120k de `overfit`**. Es legítimo porque las configs son
idénticas en las 34 claves que no son `num_steps` ni rutas, pero se dice, y el solapamiento en 60k da el piso de
no-determinismo con el que hay que leer cualquier diferencia (lo cuantifica el notebook 08).

Los datasets salen del YAML: `data.root` (train, 8839) y `data.val_root` (held-out, 465). El **examen fijo de train** son las
primeras 465 del orden canónico, igual que en el loop.

In [ ]:
# --- Resolver las dos fuentes y descubrir sus checkpoints ---
FUENTES = [
    {"clave": "base",    "config": "cats_vp.yaml",            "env": "CATS_VP"},
    {"clave": "overfit", "config": "cats_vp-overfitting.yaml", "env": "CATS_VP_OVERFIT"},
]
Ckpt = namedtuple("Ckpt", "fuente paso ruta meta")

CFG = None
for f in FUENTES:
    cfg_path = _root / "config" / f["config"]
    ckpt_target = None
    if cfg_path.exists():
        with open(cfg_path, "r", encoding="utf-8") as fh:
            f["cfg"] = yaml.safe_load(fh)
        ckpt_target = pathlib.Path(f["cfg"]["out"]["checkpoint"])
        CFG = CFG or f["cfg"]
    if os.environ.get(f"{f['env']}_CKPT"):
        ckpt_target = pathlib.Path(os.environ[f"{f['env']}_CKPT"])
    if ckpt_target is None:
        raise FileNotFoundError(f"Fuente '{f['clave']}': falta {cfg_path} y {f['env']}_CKPT")
    f["ckpt_dir"], f["stem"] = ckpt_target.parent, ckpt_target.stem

if CFG is None:
    raise FileNotFoundError("No pude leer ningún YAML: necesito data.root/val_root e image_size")

TODOS = []
for f in FUENTES:
    d = f["ckpt_dir"]
    if not d.is_dir():
        raise FileNotFoundError(f"Fuente '{f['clave']}': no existe {d}")
    for p in sorted(d.glob(f"{f['stem']}*.pt")):
        if p.name.endswith(".resume.pt") or p.stem.endswith("_raw"):
            continue                                  # sidecar de resume / hermano de crudos
        try:
            _sd, meta = load_checkpoint(p, map_location="cpu")
        except (KeyError, RuntimeError) as exc:
            print(f"  (salteado {p.name}: {exc})")
            continue
        hist = meta.get("history") or []
        m = re.search(r"_step(\d+)", p.stem)
        paso = len(hist) if hist else (int(m.group(1)) if m else -1)
        TODOS.append(Ckpt(f["clave"], paso, p, meta))

orden = {f["clave"]: i for i, f in enumerate(FUENTES)}
TODOS.sort(key=lambda c: (c.paso, orden[c.fuente]))
# Serie temporal única: un checkpoint por paso (si está en las dos fuentes, la primera declarada).
SERIE, _vistos = [], set()
for c in TODOS:
    if c.paso not in _vistos:
        SERIE.append(c)
        _vistos.add(c.paso)

print(f"checkpoints descubiertos: {len(TODOS)} | serie única por paso: {len(SERIE)}")
for c in TODOS:
    en_serie = "*" if c in SERIE else " "
    print(f"  {en_serie} {c.fuente:9s} paso {c.paso:>7,} -> {c.ruta.name}")


def buscar(paso):
    """El checkpoint de un paso exacto; falla con la lista disponible si no está."""
    hallado = [c for c in SERIE if c.paso == paso]
    if not hallado:
        raise KeyError(
            f"No hay checkpoint en el paso {paso:,}. Disponibles: "
            f"{[c.paso for c in SERIE]}. Con keep_last_checkpoints=4 los snapshots viejos de "
            "la corrida larga se borran: revisá qué pasos pediste en PASOS_FIG3/PASOS_FIG4."
        )
    return hallado[0]


# --- Datasets y geometría ---
DATA = CFG["data"]
IMAGE_SIZE = int(DATA["image_size"])
T_EPS = float(CFG["train"].get("t_eps", 1e-4))
SDE_NAME = CFG["sde"]["name"]
FORMA = (3, IMAGE_SIZE, IMAGE_SIZE)
D_PIX = int(np.prod(FORMA))

TRAIN_ROOT = pathlib.Path(os.environ.get("CATS_TRAIN_ROOT", _root / DATA["root"]))
VAL_ROOT = pathlib.Path(os.environ.get("CATS_VAL_ROOT", _root / DATA["val_root"]))
for nom, r in (("train", TRAIN_ROOT), ("val", VAL_ROOT)):
    if not r.is_dir():
        raise FileNotFoundError(
            f"No existe la carpeta de {nom}: {r}. Corré el notebook en la máquina donde vive "
            "el dataset, o exportá CATS_TRAIN_ROOT / CATS_VAL_ROOT."
        )
N_TRAIN, N_VAL = count_images(TRAIN_ROOT), count_images(VAL_ROOT)

sde = make_sde(SDE_NAME, data_dim=FORMA)
print(f"\nSDE={SDE_NAME} forma={FORMA} D={D_PIX} t_eps={T_EPS}")
print(f"train: {N_TRAIN:,} en {TRAIN_ROOT}")
print(f"val  : {N_VAL:,} en {VAL_ROOT}")

# El examen fijo de train son las primeras N_VAL del orden canónico (igual que el loop).
N_EXAMEN = N_VAL
print(f"examen fijo de train: primeras {N_EXAMEN} del orden canónico")
if N_TRAIN < N_EXAMEN:
    raise ValueError(f"train ({N_TRAIN}) más chico que el examen ({N_EXAMEN})")
print("\nALINEACIÓN train↔val: los dos exámenes tienen el MISMO tamaño "
      f"({N_EXAMEN} == {N_VAL}), así que compartir t y eps índice por índice es posible "
      "-> los NIVELES de las dos curvas son comparables, no solo las pendientes.")

In [ ]:
# --- Cargar los conjuntos a memoria (una vez) y los latentes fijos ---
def leer_todo(root, max_images=None):
    """Todas las imágenes de una carpeta como un solo tensor (B,3,S,S) en orden canónico."""
    lotes = list(finite_batches(root, 128, image_size=IMAGE_SIZE, max_images=max_images,
                                num_workers=0))
    return torch.cat(lotes, 0)


t0 = time.time()
X_TRAIN = leer_todo(TRAIN_ROOT)                    # (N_TRAIN, 3, S, S) — referencia completa
X_EXAMEN = X_TRAIN[:N_EXAMEN].clone()              # examen fijo de train
X_VAL = leer_todo(VAL_ROOT)                        # held-out completo
print(f"leídas en {time.time()-t0:.1f}s | train={tuple(X_TRAIN.shape)} "
      f"examen={tuple(X_EXAMEN.shape)} val={tuple(X_VAL.shape)}")
print(f"memoria train: {X_TRAIN.numel()*4/1e9:.2f} GB en fp32")
for nom, x in (("train", X_TRAIN), ("val", X_VAL)):
    print(f"  {nom}: rango=({float(x.min()):.2f}, {float(x.max()):.2f}) "
          f"media={float(x.mean()):+.4f} std={float(x.std()):.4f}")

# --- Latentes fijos: se generan UNA vez y viven en disco ---
LAT_PATH = CACHE / f"latentes_{N_METRICAS}_{SEED_LATENTES}_{IMAGE_SIZE}.pt"
if LAT_PATH.exists() and not RECALCULAR:
    LATENTES = torch.load(LAT_PATH, map_location="cpu")
    print(f"\nlatentes fijos <- {LAT_PATH.name}")
else:
    g = torch.Generator().manual_seed(SEED_LATENTES)
    LATENTES = sde.prior_sampling((N_METRICAS, *FORMA), generator=g, device="cpu")
    torch.save(LATENTES, LAT_PATH)
    print(f"\nlatentes fijos -> {LAT_PATH.name} (nuevos)")
print(f"  {tuple(LATENTES.shape)} | los primeros 4 son las columnas de la Figura 3")
print(f"  huella sha-ish: suma={float(LATENTES.sum()):+.6f} (identifica el conjunto)")


def cargar_red(c, device=DEVICE):
    """Reconstruye la red de un checkpoint y devuelve (score_fn, red_cruda).

    Mismo camino que generate_from_checkpoint: make_model con la receta + wrap por
    score_parametrization. Los pesos publicados son la sombra EMA (meta['ema']).
    """
    state, meta = load_checkpoint(c.ruta, map_location="cpu")
    receta = meta["model"]
    cruda = make_model(receta["name"], **dict(receta["kwargs"]))
    cruda.load_state_dict(state)
    red = cruda
    if receta.get("score_parametrization") == "epsilon":
        red = EpsilonScoreWrapper(cruda, lambda x, t: sde.marginal_prob(x, t)[1])
    red.eval().to(device)
    return red


def alpha_sigma(t, device=DEVICE):
    """(alpha_t, sigma_t) escalares por muestra, shape (B,1). Genérico para cualquier SDE."""
    t = t.to(device)
    uno = torch.ones((t.shape[0], *FORMA), device=device)
    a = sde.marginal_prob(uno, t)[0].flatten(1)[:, :1]
    s = sde.marginal_prob(uno, t)[1].flatten(1)[:, :1]
    return a, s


_a, _s = alpha_sigma(torch.tensor([T_EPS, 0.5, 1.0]))
print("\nalpha/sigma de control:")
for tv, a, s in zip([T_EPS, 0.5, 1.0], _a.flatten().tolist(), _s.flatten().tolist()):
    print(f"  t={tv:<7.4g} alpha={a:.6f} sigma={s:.6f}")

## Figura 1 — Loss DSM: agregada y por decil de $t$

**Panel izquierdo (la motivación):** el promedio sobre todos los $t$, para el examen fijo de train y para el held-out. Es la
curva que ya conocés, recomputada acá con el grid estratificado.

**Panel derecho (el diagnóstico):** la val loss promediada **dentro de cada decil de $t$**, una curva por decil, contra el paso.
Como la loss varía en órdenes de magnitud a lo largo de $t$ —a $t$ grande la tarea es casi trivial, a $t$ chico casi
imposible—, cada decil va **normalizado por su propio valor inicial**. Sin eso las curvas de $t$ alto quedan aplastadas contra
el cero y no se ve nada.

**Lo que se espera:** que la subida esté concentrada en los deciles **bajos y medios**, donde vive el detalle fino y donde
memorizar paga. Si los deciles altos siguen planos, queda confirmado que no hay degradación global sino un fenómeno
**localizado en $t$**.

**El examen:** un $t_{i,d}$ por (imagen $i$, decil $d$) sorteado dentro del bin, y un $\varepsilon_{i,d}$ fijo — **los mismos
para train y para val, índice por índice**. La pérdida por muestra usa `sde.score_target` directamente (que devuelve
$s^\ast=-\varepsilon/\sigma$ y $\lambda=\sigma^2$, o sea $\|\sigma s_\theta+\varepsilon\|^2$), así el número es el mismo
objetivo que se optimizó.

In [ ]:
# --- Grid estratificado de t y el ruido fijo del examen ---
def bordes_deciles():
    if ESCALA_T == "log":
        return np.geomspace(T_EPS, sde.T, N_DECILES + 1)
    return np.linspace(T_EPS, sde.T, N_DECILES + 1)


BORDES = bordes_deciles()
CENTROS = np.sqrt(BORDES[:-1] * BORDES[1:]) if ESCALA_T == "log" else 0.5 * (BORDES[:-1] + BORDES[1:])

_g = torch.Generator().manual_seed(SEED_EXAMEN)
# t[d, i]: un tiempo por (decil, imagen), sorteado DENTRO del bin d.
_u = torch.rand((N_DECILES, N_EXAMEN), generator=_g).numpy()
TS = (BORDES[:-1, None] * (BORDES[1:, None] / BORDES[:-1, None]) ** _u if ESCALA_T == "log"
      else BORDES[:-1, None] + _u * (BORDES[1:, None] - BORDES[:-1, None])).astype(np.float32)
# eps[d, i]: el MISMO ruido para train y val en el índice i. Sorteado una vez.
EPS_EXAMEN = torch.randn((N_DECILES, N_EXAMEN, *FORMA), generator=_g)

print(f"deciles en escala {ESCALA_T}: bordes de {BORDES[0]:.2e} a {BORDES[-1]:.2f}")
for d in range(N_DECILES):
    print(f"  decil {d}: t ∈ [{BORDES[d]:.4g}, {BORDES[d+1]:.4g}]  centro≈{CENTROS[d]:.4g}  "
          f"t muestreados ∈ [{TS[d].min():.4g}, {TS[d].max():.4g}]")
print(f"ruido del examen: {tuple(EPS_EXAMEN.shape)} "
      f"({EPS_EXAMEN.numel()*4/1e9:.2f} GB fp32, compartido entre train y val)")


def perdidas_por_muestra(red, x0, t, eps):
    """Pérdida DSM POR MUESTRA (media sobre píxeles), idéntica al objetivo entrenado."""
    mean, std = sde.marginal_prob(x0, t)
    xt = mean + std * eps
    s_real, w = sde.score_target(x0, t, eps)
    s_pred = red(xt, t)
    return (w * (s_pred - s_real).pow(2)).flatten(1).mean(1)


def tabla_perdidas():
    """(n_ckpt, 2, N_DECILES, N_EXAMEN): pérdida por muestra. split 0=train(examen), 1=val."""
    conjuntos = [X_EXAMEN, X_VAL[:N_EXAMEN]]
    out = np.zeros((len(SERIE), 2, N_DECILES, N_EXAMEN), dtype=np.float32)
    for ci, c in enumerate(SERIE):
        red = cargar_red(c)
        t0 = time.time()
        with torch.no_grad():
            for si, X in enumerate(conjuntos):
                for d in range(N_DECILES):
                    for i0 in range(0, N_EXAMEN, 128):
                        i1 = min(i0 + 128, N_EXAMEN)
                        x0 = X[i0:i1].to(DEVICE)
                        t = torch.from_numpy(TS[d, i0:i1]).to(DEVICE)
                        eps = EPS_EXAMEN[d, i0:i1].to(DEVICE)
                        out[ci, si, d, i0:i1] = (
                            perdidas_por_muestra(red, x0, t, eps).cpu().numpy())
        del red
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
        print(f"    paso {c.paso:>7,} ({c.fuente}) {time.time()-t0:5.1f}s")
    return {"perdidas": out, "pasos": np.array([c.paso for c in SERIE]),
            "ts": TS, "bordes": BORDES.astype(np.float32)}


print()
_d = cacheado(f"fig1_perdidas_{ESCALA_T}_{N_DECILES}_{SEED_EXAMEN}", tabla_perdidas)
PERDIDAS, PASOS = _d["perdidas"], _d["pasos"]
print(f"tabla: {PERDIDAS.shape} (ckpt, split, decil, imagen)")

# Forma larga para quien quiera cargarla en pandas: pd.DataFrame(**np.load(...))
_ck, _sp, _dc, _im = np.meshgrid(np.arange(len(PASOS)), np.arange(2), np.arange(N_DECILES),
                                 np.arange(N_EXAMEN), indexing="ij")
np.savez_compressed(
    CACHE / "fig1_tabla_larga.npz",
    paso=PASOS[_ck].ravel(), split=_sp.ravel(), decil=_dc.ravel(),
    t=TS[_dc.ravel(), _im.ravel()], loss=PERDIDAS.ravel(), imagen=_im.ravel(),
)
print(f"forma larga -> fig1_tabla_larga.npz ({PERDIDAS.size:,} filas; split 0=train 1=val)")

In [ ]:
# --- Figura 1 ---
agregada = PERDIDAS.mean(axis=(2, 3))          # (n_ckpt, 2)
por_decil = PERDIDAS.mean(axis=3)              # (n_ckpt, 2, N_DECILES)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.5, 5.2))

ax1.plot(PASOS, agregada[:, 0], marker="o", ms=4, color="C2", label="train (examen fijo)")
ax1.plot(PASOS, agregada[:, 1], marker="s", ms=4, color="C0", label="val (held-out)")
i_min = int(np.argmin(agregada[:, 1]))
ax1.axvline(PASOS[i_min], color="crimson", ls="--", lw=1.1,
            label=f"mínimo de val: {PASOS[i_min]:,}")
ax1.set_xlabel("paso de entrenamiento"); ax1.set_ylabel("pérdida DSM (promedio sobre todo $t$)")
ax1.set_title("Agregada — la motivación", fontsize=11)
ax1.legend(fontsize="small")

cmap = plt.get_cmap("viridis")
for d in range(N_DECILES):
    y = por_decil[:, 1, d] / por_decil[0, 1, d]          # normalizado por su valor inicial
    ax2.plot(PASOS, y, marker="o", ms=3, color=cmap(d / (N_DECILES - 1)),
             label=f"d{d}: t∈[{BORDES[d]:.3g}, {BORDES[d+1]:.3g}]")
ax2.axhline(1.0, color="0.3", lw=1.0)
ax2.axvline(PASOS[i_min], color="crimson", ls="--", lw=1.1)
ax2.set_xlabel("paso de entrenamiento")
ax2.set_ylabel("val loss del decil / su valor inicial")
ax2.set_title(f"Val loss por decil de $t$ (escala {ESCALA_T}), normalizada — el diagnóstico",
              fontsize=11)
ax2.legend(fontsize=6.5, ncol=2)
fig.tight_layout()
plt.show()

print(f"{'decil':6s} {'rango de t':>22s} {'val inicial':>12s} {'val final':>11s} "
      f"{'peor/mejor':>11s} {'sube desde':>11s}")
for d in range(N_DECILES):
    serie = por_decil[:, 1, d]
    j = int(np.argmin(serie))
    print(f"d{d:<5d} [{BORDES[d]:>9.4g}, {BORDES[d+1]:<9.4g}] {serie[0]:12.5g} "
          f"{serie[-1]:11.5g} {serie[-1]/serie[j]:11.4f} "
          f"{PASOS[j]:>10,}" + ("" if j < len(PASOS) - 1 else "  (no sube)"))
print()
print(f"agregada: mínimo de val en el paso {PASOS[i_min]:,} "
      f"({agregada[i_min,1]:.6g}); final {agregada[-1,1]:.6g} "
      f"(+{100*(agregada[-1,1]/agregada[i_min,1]-1):.1f}%)")
print(f"gap final val-train: {agregada[-1,1]-agregada[-1,0]:+.6g}  "
      f"(inicial {agregada[0,1]-agregada[0,0]:+.6g})")
print("Los niveles son comparables: mismo t y mismo eps índice por índice en los dos exámenes.")

## Figura 2 — $E(t)$ contra el score empírico

La figura central. Paso de entrenamiento en $x$, $E(t)$ en $y$, log-log. Dos paneles —$\hat x_0^{\text{emp}}$ del **train** y
del **held-out**—, una curva por decil de $t$ en cada uno.

**Cómo se lee:**

- **Panel train:** $E$ decreciendo monótonamente = el modelo converge al óptimo del *training set*. Eso **es** memorización,
  medida directamente, sin proxies.
- **Panel held-out:** si tiene mínimo y cae cerca de los 30k, el argumento cierra: la val loss y esta curva marcan el mismo
  punto por **dos caminos independientes**.

**Dónde se evalúa la esperanza (el detalle que hace o rompe la figura):** se muestrea
$x=\bar\alpha_t x_0+\sigma_t\varepsilon$ con $x_0$ **del held-out en los dos paneles**. Si se evaluara cerca de los modos de
train, el score empírico de train tendría ventaja por construcción. La arena es neutral.

**Numérico:** los pesos van por `logsumexp`/`softmax` con resta del máximo — a $t$ chico los exponentes se van a $-10^4$ y en
float32 sin eso explota. La celda **verifica la identidad de Tweedie** contra autograd antes de medir nada, y reporta
$\max_i w_i$ por decil: cuando ese número se acerca a 1, $\hat x_0^{\text{emp}}$ **ya es** el vecino más cercano, que es el
límite $t\to0$ y el puente con la Figura 4.

In [ ]:
# --- Núcleo: x0 predicho por la red y x0 empírico de un conjunto de referencia ---
def x0_theta(red, xt, t):
    """x̂0^θ = (x + σ² s_θ)/ᾱ  — Tweedie sobre el score que publica la red."""
    a, s = alpha_sigma(t)
    a = a.view(-1, *([1] * len(FORMA)))
    s = s.view(-1, *([1] * len(FORMA)))
    return (xt + s.pow(2) * red(xt, t)) / a.clamp_min(1e-8)


def x0_emp(xt, t, ref_flat, ref_sq):
    """x̂0^emp = Σ_i w_i x0_i con w_i ∝ exp(-‖x-ᾱx0_i‖²/2σ²). Devuelve (x̂0, max_i w_i).

    ‖x-ᾱx0_i‖² se expande como ‖x‖² - 2ᾱ⟨x,x0_i⟩ + ᾱ²‖x0_i‖² para no materializar la
    diferencia. El softmax va en float64 y resta el máximo: sin eso, a σ chico los
    exponentes (~-1e4 y peores) desbordan float32.
    """
    a, s = alpha_sigma(t)
    q = xt.flatten(1)
    d2 = ((q * q).sum(1, keepdim=True)
          - 2.0 * a * (q @ ref_flat.T)
          + a.pow(2) * ref_sq[None, :])
    w = torch.softmax((-d2 / (2.0 * s.pow(2)).clamp_min(1e-12)).double(), dim=1)
    return (w @ ref_flat.double()).float().view(-1, *FORMA), w.max(1).values.float()


def preparar_ref(X):
    f = X.flatten(1).to(DEVICE)
    return f, (f * f).sum(1)


# x0_theta tiene que coincidir con el paso final de denoising del sampler (_denoise):
# es la misma formula de Tweedie. Si divergen, una de las dos esta mal.
def _chequear_denoise():
    red = cargar_red(SERIE[-1])
    smp = make_sampler(SAMPLER_FIG, sde, red, n_steps=PASOS_FIG, t_eps=T_EPS,
                       time_grid=GRILLA_T, **kwargs_para(SAMPLER_FIG))
    t = torch.full((4,), 0.3, device=DEVICE)
    x = torch.randn((4, *FORMA), device=DEVICE)
    with torch.no_grad():
        a = x0_theta(red, x, t)
        b = smp._denoise(x, t)
    rel = float((a - b).norm() / b.norm())
    del red, smp
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return rel


_rel = _chequear_denoise()
print(f"x0_theta vs el paso final de denoising del sampler: error relativo = {_rel:.2e}")
if _rel > 1e-5:
    raise AssertionError("x0_theta y sampler._denoise discrepan: revisa alpha_sigma")
print("(el objeto central de la Figura 2 ES la ultima operacion que hace el sampler)")


# --- Verificación de Tweedie: el score de la mixtura por autograd vs -(x-ᾱx̂0)/σ² ---
# TODO el chequeo va en float64, incluido el camino de autograd: en float32 el propio
# gradiente de referencia tiene error ~1e-4 y el número impreso mediría la precisión de la
# referencia en vez del acuerdo entre las dos expresiones, que es lo que se quiere verificar.
_sub = X_VAL[:16].to(DEVICE).double()
_rf64 = X_TRAIN[:512].flatten(1).to(DEVICE).double()
_rs64 = (_rf64 * _rf64).sum(1)
_rf32, _rs32 = preparar_ref(X_TRAIN[:512])
print("verificación de Tweedie (score por autograd del logsumexp vs -(x-ᾱ·x̂0_emp)/σ²),")
print("todo en float64 para que el número mida el acuerdo y no la precisión de la referencia:")
for _tv in (0.9, 0.5, 0.2, 0.05):
    _t = torch.full((16,), _tv, device=DEVICE)
    _a, _s = alpha_sigma(_t)
    _a64, _s64 = _a.double(), _s.double()
    _x = (_a64.view(-1, 1, 1, 1) * _sub
          + _s64.view(-1, 1, 1, 1) * torch.randn(_sub.shape, device=DEVICE).double()
          ).requires_grad_(True)
    _q = _x.flatten(1)
    _d2 = ((_q * _q).sum(1, keepdim=True) - 2 * _a64 * (_q @ _rf64.T) + _a64.pow(2) * _rs64[None])
    _lp = torch.logsumexp(-_d2 / (2 * _s64.pow(2)), dim=1).sum()
    _g, = torch.autograd.grad(_lp, _x)
    with torch.no_grad():
        _xe, _ = x0_emp(_x.detach().float(), _t, _rf32, _rs32)
        _tw = -(_x.detach() - _a64.view(-1, 1, 1, 1) * _xe.double()) / _s64.view(-1, 1, 1, 1).pow(2)
    print(f"  t={_tv:.2f} σ={float(_s[0]):.4f}  error relativo = "
          f"{float((_g - _tw).norm() / _tw.norm()):.2e}")
del _x, _g, _rf64, _rs64, _rf32, _rs32, _sub
if DEVICE == "cuda":
    torch.cuda.empty_cache()

In [ ]:
# --- E(t) por checkpoint, decil y conjunto de referencia ---
def calcular_E():
    """num/den/wmax con shape (n_ckpt, 2, N_DECILES); ref 0=train, 1=held-out."""
    # La arena es NEUTRAL: x sale SIEMPRE del held-out, en los dos paneles.
    g = torch.Generator().manual_seed(SEED_EVAL_E)
    idx = torch.randint(0, X_VAL.shape[0], (N_EVAL_E,), generator=g)
    x0_eval = X_VAL[idx].to(DEVICE)
    eps_eval = torch.randn((N_DECILES, N_EVAL_E, *FORMA), generator=g)
    t_eval = torch.from_numpy(TS[:, :N_EVAL_E].copy()) if N_EVAL_E <= N_EXAMEN else None
    if t_eval is None:                              # más puntos que imágenes del examen
        u = torch.rand((N_DECILES, N_EVAL_E), generator=g).numpy()
        t_eval = torch.from_numpy(
            (BORDES[:-1, None] * (BORDES[1:, None] / BORDES[:-1, None]) ** u
             if ESCALA_T == "log"
             else BORDES[:-1, None] + u * (BORDES[1:, None] - BORDES[:-1, None])
             ).astype(np.float32))

    refs = [preparar_ref(X_TRAIN), preparar_ref(X_VAL)]
    num = np.zeros((len(SERIE), 2, N_DECILES), dtype=np.float64)
    den = np.zeros((len(SERIE), 2, N_DECILES), dtype=np.float64)
    wmx = np.zeros((2, N_DECILES), dtype=np.float64)

    for ci, c in enumerate(SERIE):
        red = cargar_red(c)
        t0 = time.time()
        with torch.no_grad():
            for d in range(N_DECILES):
                for i0 in range(0, N_EVAL_E, LOTE_DIST):
                    i1 = min(i0 + LOTE_DIST, N_EVAL_E)
                    t = t_eval[d, i0:i1].to(DEVICE)
                    a, s = alpha_sigma(t)
                    xt = (a.view(-1, 1, 1, 1) * x0_eval[i0:i1]
                          + s.view(-1, 1, 1, 1) * eps_eval[d, i0:i1].to(DEVICE))
                    xhat_t = x0_theta(red, xt, t)                    # una sola vez por (ckpt,d)
                    for ri, (rf, rs) in enumerate(refs):
                        xhat_e, wm = x0_emp(xt, t, rf, rs)
                        num[ci, ri, d] += float((xhat_t - xhat_e).flatten(1).pow(2).sum())
                        den[ci, ri, d] += float(xhat_e.flatten(1).pow(2).sum())
                        if ci == 0:
                            wmx[ri, d] += float(wm.sum())
        del red
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
        print(f"    paso {c.paso:>7,} ({c.fuente}) {time.time()-t0:5.1f}s")
    return {"num": num, "den": den, "wmax": wmx / N_EVAL_E,
            "pasos": np.array([c.paso for c in SERIE])}


print("E(t):")
_e = cacheado(f"fig2_E_{ESCALA_T}_{N_DECILES}_{N_EVAL_E}_{SEED_EVAL_E}", calcular_E)
NUM, DEN, WMAX = _e["num"], _e["den"], _e["wmax"]
E = NUM / DEN

print(f"\n{'decil':6s} {'rango de t':>22s} {'max_i w_i (train)':>18s} {'régimen':>26s}")
n_nn = 0
for d in range(N_DECILES):
    wm = WMAX[0, d]
    reg = ("promedio genuino" if wm < 0.1 else
           "mezcla de pocos vecinos" if wm < 0.9 else "≈ vecino más cercano")
    n_nn += wm >= 0.9
    print(f"d{d:<5d} [{BORDES[d]:>9.4g}, {BORDES[d+1]:<9.4g}] {wm:18.6f} {reg:>26s}")

print(f"\n{n_nn}/{N_DECILES} deciles están en régimen de vecino más cercano (max_i w_i ≥ 0.9).")
print("Esto es MÁS ancho que el 't→0' del plan, y no es un bug: en dimensión D=%d las" % D_PIX)
print("distancias ‖x-ᾱx0_i‖² son de orden D·var, así que dividirlas por 2σ² da exponentes")
print("enormes y la diferencia entre el primer y el segundo vecino ya alcanza para saturar")
print("el softmax. Consecuencia para el texto, en dos direcciones:")
print("  · A FAVOR: 'converger al score empírico' es, en esos deciles, converger a un")
print("    denoiser de vecino más cercano. Es memorización en su forma más cruda, y la")
print("    Figura 4 mide exactamente eso — no es un apéndice, es el mismo objeto.")
print("  · EN CONTRA: esos deciles no miden cosas independientes entre sí. Los deciles con")
print("    max_i w_i bajo son los únicos donde x̂0_emp es un promedio genuino sobre el")
print("    dataset, y son los que aportan información distinta de la Figura 4.")

In [ ]:
# --- Figura 2 ---
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.4), sharey=True)
titulos = ["$\\hat x_0^{emp}$ del TRAIN — decrecer = memorizar",
           "$\\hat x_0^{emp}$ del HELD-OUT — importa la FORMA, no el nivel"]
for ri, (ax, ttl) in enumerate(zip(axes, titulos)):
    for d in range(N_DECILES):
        ax.plot(PASOS, E[:, ri, d], marker="o", ms=3, color=cmap(d / (N_DECILES - 1)),
                label=f"d{d}: t∈[{BORDES[d]:.3g}, {BORDES[d+1]:.3g}]")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("paso de entrenamiento")
    ax.set_title(ttl, fontsize=10.5)
    for d in range(N_DECILES):
        j = int(np.argmin(E[:, ri, d]))
        if 0 < j < len(PASOS) - 1:
            ax.plot([PASOS[j]], [E[j, ri, d]], marker="v", ms=7, mfc="none",
                    color=cmap(d / (N_DECILES - 1)))
axes[0].set_ylabel("$E(t)$  (relativo, log)")
axes[0].legend(fontsize=6.5, ncol=2)
axes[1].plot([], [], marker="v", ms=7, mfc="none", color="0.3", label="mínimo interior")
axes[1].legend(fontsize=7, loc="best")
fig.suptitle("$E(t)=\\mathbb{E}\\|\\hat x_0^{\\theta}-\\hat x_0^{emp}\\|^2 / "
             "\\mathbb{E}\\|\\hat x_0^{emp}\\|^2$ — x muestreado del HELD-OUT en los dos paneles",
             y=1.02, fontsize=11.5)
fig.tight_layout()
plt.show()

print("Monotonía y mínimos por decil:")
print(f"{'decil':6s} {'train: E ini -> fin':>26s} {'monótona?':>10s} "
      f"{'held-out: mínimo en':>21s} {'E min':>11s}")
for d in range(N_DECILES):
    et, eh = E[:, 0, d], E[:, 1, d]
    baja = bool(np.all(np.diff(et) <= 1e-12))
    j = int(np.argmin(eh))
    donde = f"{PASOS[j]:,}" + ("" if 0 < j < len(PASOS) - 1 else " (borde)")
    print(f"d{d:<5d} {et[0]:11.4g} -> {et[-1]:<11.4g} {str(baja):>10s} {donde:>21s} "
          f"{eh[j]:11.4g}")

_mins = [PASOS[int(np.argmin(E[:, 1, d]))] for d in range(N_DECILES)
         if 0 < int(np.argmin(E[:, 1, d])) < len(PASOS) - 1]
print()
if _mins:
    print(f"Deciles con mínimo INTERIOR en held-out: {len(_mins)}/{N_DECILES}, "
          f"mediana del paso = {int(np.median(_mins)):,}")
    print(f"(el mínimo de la val loss agregada cayó en {PASOS[i_min]:,} — dos caminos "
          "independientes marcando el mismo punto)")
else:
    print("Ningún decil tiene mínimo interior en held-out: E baja o sube en todo el rango.")
print()
print("RECORDATORIO para el texto: el NIVEL del panel held-out no es interpretable. El score")
print("empírico del held-out es otra aproximación finita, así que E_heldout ≈ E_poblacional +")
print("un offset independiente del checkpoint. El mínimo está en el lugar correcto; el valor no.")

## Figura 3 — Muestras a 15k / 30k / 120k, latentes fijos

**Qué contesta:** ¿el modelo con peor val loss genera peor? Esa es la disociación.

Grilla 3×4: filas = checkpoints, columnas = **los primeros 4 latentes** de los `N_METRICAS` fijos.
`heun` @ `NFE_FIG` NFE, pesos EMA, todo idéntico salvo el checkpoint.

**Por qué esos tres:** 15k es el control de submuestreo —algunos latentes ni se comprometen con un modo y sale una imagen
degenerada, lo cual muestra que el pipeline **detecta** un modelo malo—. 30k vs 120k es el contraste real. 45k y 60k salen
porque son visualmente indistinguibles de 30k y solo diluyen.

**Lo que la figura no puede probar:** con 4 muestras no se puede decir nada sobre diversidad ni sobre calidad promedio. Es
evidencia de que **no hay degradación visible**, que es todo lo que el argumento necesita.

`interpolation="nearest"` en todos los `imshow`: la interpolación bilineal suaviza justo el grano que hay que mostrar a
64×64.

In [ ]:
# --- Generación con latentes fijos ---
def generar(c, latentes, lote=LOTE_GEN):
    """Muestrea desde latentes FIJOS con SAMPLER_FIG a NFE_FIG. Devuelve (N,3,S,S) en CPU."""
    red = cargar_red(c)
    smp = make_sampler(SAMPLER_FIG, sde, red, n_steps=PASOS_FIG, t_eps=T_EPS,
                       time_grid=GRILLA_T, **kwargs_para(SAMPLER_FIG))
    salidas = []
    with torch.no_grad():
        for i0 in range(0, latentes.shape[0], lote):
            xT = latentes[i0:i0 + lote].to(DEVICE)
            g = torch.Generator(device=DEVICE).manual_seed(0)   # heun es determinista
            salidas.append(smp.sample(n_samples=xT.shape[0], init=xT, generator=g).cpu())
    del red, smp
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return torch.cat(salidas, 0)


CKPT_FIG3 = [buscar(p) for p in PASOS_FIG3]
print(f"Figura 3: {SAMPLER_FIG} @ {NFE_EFECTIVO} NFE ({PASOS_FIG} pasos, grilla "
      f"'{GRILLA_T}'), 4 latentes fijos")
muestras3 = {}
for c in CKPT_FIG3:
    t0 = time.time()
    muestras3[c.paso] = generar(c, LATENTES[:4])
    x = muestras3[c.paso]
    print(f"  {c.fuente:9s} paso {c.paso:>7,} {time.time()-t0:5.1f}s  "
          f"rango=({float(x.min()):+.2f}, {float(x.max()):+.2f}) "
          f"finito={bool(torch.isfinite(x).all())}")

fig, axes = plt.subplots(len(CKPT_FIG3), 4, figsize=(4 * 2.3, len(CKPT_FIG3) * 2.35),
                         squeeze=False)
for r, c in enumerate(CKPT_FIG3):
    arr = denorm(muestras3[c.paso])
    for k in range(4):
        ax = axes[r][k]
        ax.imshow(arr[k], interpolation="nearest")
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        if r == 0:
            ax.set_title(f"latente {k}", fontsize=9)
        if k == 0:
            ax.set_ylabel(f"{c.paso//1000}k\n({c.fuente})", fontsize=9.5)
fig.suptitle(f"Muestras no seleccionadas · latentes fijados a priori y reusados en todas las "
             f"figuras\n{SAMPLER_FIG} @ {NFE_EFECTIVO} NFE (grilla '{GRILLA_T}'), pesos EMA — solo cambia el checkpoint",
             y=1.005, fontsize=10.5)
fig.tight_layout()
fig.savefig(CACHE / "figura3_muestras.png", dpi=220, bbox_inches="tight")
plt.show()
print(f"PNG a 220 dpi -> {CACHE / 'figura3_muestras.png'}")
print("Caption obligatorio: muestras NO seleccionadas; los latentes se fijaron a priori")
print("(semilla {}) y se reusan en todas las figuras.".format(SEED_LATENTES))

## Figura 4 — Vecinos más cercanos + distribuciones

**Qué contesta:** las muestras de 120k, ¿son copias del training set?

**Panel A (cualitativo):** dos filas, una por checkpoint. Cada fila: una muestra generada y sus 3 vecinos más cercanos del
train. Se elige **la muestra con menor distancia al NN** en cada checkpoint — es el **peor caso**, no un ejemplo típico, y se
declara.

**Panel B (cuantitativo):** histogramas superpuestos de la distancia al NN, con cuatro distribuciones:

| distribución | rol |
|---|---|
| held-out → train | **el nulo**: cuán cerca están dos gatos reales distintos |
| train → train (excl. sí misma) | el piso: densidad del dataset |
| muestras 30k → train | |
| muestras 120k → train | |

**Por qué el nulo no es opcional:** el dataset es de una sola clase, con caras alineadas y centradas. Dos gatos distintos ya
están cerca. Sin el nulo, cualquier distancia reportada es no interpretable. La conclusión **no** es "la distancia es baja",
es "la distribución de 120k está corrida a la izquierda respecto del nulo, y más que la de 30k".

**La métrica:** euclídea sobre imágenes normalizadas por muestra (restar media, dividir por std), dividida por $\sqrt{2D}$.
Eso la vuelve exactamente $d=\sqrt{1-\rho}\in[0,\sqrt2]$, con 0 = idéntica, y saca de la ecuación el brillo y el contraste
globales. Nada más: sin LPIPS, sin redes preentrenadas.

**La búsqueda va sobre $\{$identidad, hflip$\}$** — el entrenamiento usa `augment: true` (volteo horizontal aleatorio,
verificado en el YAML), así que la memorización aparece **espejada** y contra la original la distancia daría grande. La función
devuelve el índice de la imagen **original** aunque el match haya sido con la espejada.

**Al texto:** la mediana de cada distribución y un KS de dos muestras entre 30k y 120k.

In [ ]:
# --- Generar las N_METRICAS muestras de cada checkpoint (cacheado: es lo más caro) ---
print(f"augment en el YAML: {DATA.get('augment')} -> la búsqueda con hflip es obligatoria")
CKPT_FIG4 = [buscar(p) for p in PASOS_FIG4]

MUESTRAS4 = {}
for c in CKPT_FIG4:
    p = (CACHE / f"muestras_{c.fuente}_{c.paso}_{SAMPLER_FIG}{NFE_EFECTIVO}"
         f"_{GRILLA_T}_{N_METRICAS}.pt")
    if p.exists() and not RECALCULAR:
        MUESTRAS4[c.paso] = torch.load(p, map_location="cpu")
        print(f"  [cache] paso {c.paso:>7,} <- {p.name}")
    else:
        t0 = time.time()
        MUESTRAS4[c.paso] = generar(c, LATENTES)
        torch.save(MUESTRAS4[c.paso], p)
        print(f"  [calc ] paso {c.paso:>7,} {N_METRICAS} muestras en "
              f"{(time.time()-t0)/60:.1f} min -> {p.name}")
    x = MUESTRAS4[c.paso]
    print(f"          {tuple(x.shape)} rango=({float(x.min()):+.2f}, {float(x.max()):+.2f}) "
          f"finito={bool(torch.isfinite(x).all())}")


# --- La métrica: d = ‖â-b̂‖/√(2D) = √(1-ρ) ∈ [0,√2] ---
def normalizar(X):
    """Por muestra: restar media y dividir por std. Deja ‖â‖² = D."""
    f = X.flatten(1).float()
    f = f - f.mean(1, keepdim=True)
    return f / f.std(1, unbiased=False, keepdim=True).clamp_min(1e-8)


def vecinos(consultas, referencia, k=1, excluir_diagonal=False):
    """(dist, idx) a los k vecinos más cercanos del referencia, sobre {identidad, hflip}.

    `idx` es SIEMPRE el índice de la imagen ORIGINAL, incluso si el match fue con su
    espejada. Con excluir_diagonal se descarta la imagen i contra sí misma Y contra su
    propio espejo (si no, un gato simétrico da d≈0 y contamina el piso del dataset).
    """
    A = normalizar(consultas).to(DEVICE)
    B = normalizar(referencia).to(DEVICE)
    B_flip = normalizar(torch.flip(referencia, dims=[3])).to(DEVICE)
    n, D = A.shape[0], A.shape[1]
    dist = torch.empty((n, k), device=DEVICE)
    idx = torch.empty((n, k), dtype=torch.long, device=DEVICE)
    for i0 in range(0, n, LOTE_DIST):
        i1 = min(i0 + LOTE_DIST, n)
        a = A[i0:i1]
        d_id = (1.0 - (a @ B.T) / D).clamp_min(0.0)
        d_fl = (1.0 - (a @ B_flip.T) / D).clamp_min(0.0)
        d = torch.minimum(d_id, d_fl).sqrt()             # √(1-ρ), el mejor de los dos
        if excluir_diagonal:
            filas = torch.arange(i0, i1, device=DEVICE)
            d[filas - i0, filas] = float("inf")
        vd, vi = torch.topk(d, k, dim=1, largest=False)
        dist[i0:i1], idx[i0:i1] = vd, vi
    return dist.cpu().numpy(), idx.cpu().numpy()


print("\ncalculando distancias al vecino más cercano (identidad + hflip)...")
t0 = time.time()
d_nulo, _ = vecinos(X_VAL, X_TRAIN, k=1)                              # el nulo
d_piso, _ = vecinos(X_TRAIN, X_TRAIN, k=1, excluir_diagonal=True)     # densidad del dataset
D_NN, IDX_NN = {}, {}
for c in CKPT_FIG4:
    D_NN[c.paso], IDX_NN[c.paso] = vecinos(MUESTRAS4[c.paso], X_TRAIN, k=N_VECINOS)
print(f"  {time.time()-t0:.1f}s")

DISTRIB = [
    ("held-out → train (EL NULO)", d_nulo[:, 0], "C7", "-"),
    ("train → train (excl. sí misma)", d_piso[:, 0], "0.35", "--"),
]
for c, col in zip(CKPT_FIG4, ["C0", "C3"]):
    DISTRIB.append((f"muestras {c.paso//1000}k → train", D_NN[c.paso][:, 0], col, "-"))

print(f"\n{'distribución':34s} {'n':>6s} {'mediana':>9s} {'p5':>8s} {'p25':>8s} {'mín':>8s}")
for nom, v, _c, _ls in DISTRIB:
    print(f"{nom:34s} {len(v):6d} {np.median(v):9.4f} {np.percentile(v,5):8.4f} "
          f"{np.percentile(v,25):8.4f} {v.min():8.4f}")

ks = ks_2samp(D_NN[PASOS_FIG4[0]][:, 0], D_NN[PASOS_FIG4[1]][:, 0])
print(f"\nKS de dos muestras {PASOS_FIG4[0]//1000}k vs {PASOS_FIG4[1]//1000}k: "
      f"D={ks.statistic:.4f}  p={ks.pvalue:.3e}")
_m0, _m1 = (np.median(D_NN[p][:, 0]) for p in PASOS_FIG4)
_mn = np.median(d_nulo[:, 0])
print(f"medianas: nulo={_mn:.4f} | {PASOS_FIG4[0]//1000}k={_m0:.4f} | "
      f"{PASOS_FIG4[1]//1000}k={_m1:.4f}")
print(f"corrimiento respecto del nulo: {PASOS_FIG4[0]//1000}k {_m0-_mn:+.4f} | "
      f"{PASOS_FIG4[1]//1000}k {_m1-_mn:+.4f}")

In [ ]:
# --- Figura 4: panel A (peor caso) y panel B (distribuciones) ---
fig = plt.figure(figsize=(13.5, 9.2))
gs = fig.add_gridspec(2, 2, height_ratios=[1.0, 0.95], hspace=0.40, wspace=0.22,
                      top=0.875, bottom=0.07, left=0.07, right=0.97)

# Panel A: la muestra MÁS CERCANA a su NN (peor caso), con sus N_VECINOS vecinos.
# El rótulo del panel va con fig.text y NO como título de un eje que abarque la fila:
# un título de eje contenedor cae a la misma altura que los títulos de los subplots
# del subgridspec y se pisan. Las distancias van DENTRO de cada imagen por la misma
# razón — un título por fila choca con las imágenes de la fila de arriba.
fig.text(0.5, 0.905,
         "Panel A — peor caso por checkpoint: la muestra con MENOR distancia a su vecino más "
         f"cercano, y sus {N_VECINOS} vecinos del train (no es un ejemplo típico)",
         ha="center", va="bottom", fontsize=10.5)
sub = gs[0, :].subgridspec(len(CKPT_FIG4), 1 + N_VECINOS, hspace=0.10, wspace=0.06)
for r, c in enumerate(CKPT_FIG4):
    j = int(np.argmin(D_NN[c.paso][:, 0]))
    ax = fig.add_subplot(sub[r, 0])
    ax.imshow(denorm(MUESTRAS4[c.paso][j:j + 1])[0], interpolation="nearest")
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    ax.set_ylabel(f"{c.paso//1000}k\nlatente {j}", fontsize=9)
    if r == 0:
        ax.set_title("muestra generada", fontsize=9)
    for k in range(N_VECINOS):
        axv = fig.add_subplot(sub[r, 1 + k])
        axv.imshow(denorm(X_TRAIN[IDX_NN[c.paso][j, k]:IDX_NN[c.paso][j, k] + 1])[0],
                   interpolation="nearest")
        axv.set_xticks([]); axv.set_yticks([]); axv.grid(False)
        if r == 0:
            axv.set_title(f"vecino {k+1}", fontsize=9)
        axv.text(0.04, 0.04, f"d={D_NN[c.paso][j,k]:.3f}", transform=axv.transAxes,
                 fontsize=8, color="w", va="bottom",
                 bbox=dict(facecolor="k", alpha=0.6, pad=1.5, linewidth=0))

# Panel B: las cuatro distribuciones.
axB = fig.add_subplot(gs[1, 0])
bins = np.linspace(0, max(float(np.percentile(v, 99.5)) for _n, v, _c, _l in DISTRIB) * 1.05, 60)
for nom, v, col, ls in DISTRIB:
    axB.hist(v, bins=bins, density=True, histtype="step", lw=1.7, color=col, ls=ls, label=nom)
    axB.axvline(np.median(v), color=col, ls=":", lw=1.0, alpha=0.8)
axB.set_xlabel(r"distancia al vecino más cercano  $d=\sqrt{1-\rho}$")
axB.set_ylabel("densidad")
axB.set_title("Panel B — distribuciones (punteadas: medianas)", fontsize=10.5)
axB.legend(fontsize=7.5)

axC = fig.add_subplot(gs[1, 1])
for nom, v, col, ls in DISTRIB:
    xs = np.sort(v)
    axC.plot(xs, np.arange(1, len(xs) + 1) / len(xs), lw=1.7, color=col, ls=ls, label=nom)
axC.set_xlabel(r"$d=\sqrt{1-\rho}$"); axC.set_ylabel("F(d) empírica")
axC.set_title(f"Acumuladas — KS({PASOS_FIG4[0]//1000}k, {PASOS_FIG4[1]//1000}k): "
              f"D={ks.statistic:.3f}, p={ks.pvalue:.2e}", fontsize=10.5)
axC.legend(fontsize=7.5)

# Sin tight_layout: pelearía con los márgenes explícitos del gridspec y volvería a
# amontonar el panel A.
fig.suptitle("¿Son copias del training set? La respuesta es el CORRIMIENTO respecto del nulo, "
             "no el valor absoluto", y=0.965, fontsize=11.5)
fig.savefig(CACHE / "figura4_vecinos.png", dpi=220, bbox_inches="tight")
plt.show()
print(f"PNG a 220 dpi -> {CACHE / 'figura4_vecinos.png'}")

print("\nNúmeros para el texto:")
for nom, v, _c, _l in DISTRIB:
    print(f"  mediana {nom:34s} = {np.median(v):.4f}")
print(f"  KS(30k, 120k): D={ks.statistic:.4f}, p={ks.pvalue:.3e} "
      f"(n={len(D_NN[PASOS_FIG4[0]])} por checkpoint)")
print("\nLectura correcta: NO 'la distancia es baja' sino 'la distribución de "
      f"{PASOS_FIG4[1]//1000}k está corrida a la izquierda respecto del nulo, y más que la de "
      f"{PASOS_FIG4[0]//1000}k'.")

## Cierre — qué quedó probado y con qué límites

- **Figura 1** localiza la subida de la val loss **en $t$**, con un examen reconstruido con grid estratificado y con el mismo
  $t_i$ y $\varepsilon_i$ para train y val índice por índice. Los **niveles** son comparables (ya lo eran en el código: los
  dos exámenes comparten `time_sampler` y semilla, y los conjuntos tienen igual tamaño — el notebook lo verifica).
- **Figura 2** mide la distancia al **minimizador exacto** del objetivo de DSM para el dataset finito, con la identidad de
  Tweedie verificada contra autograd. El panel train decreciendo **es** memorización medida; el mínimo del panel held-out y
  el mínimo de la val loss son dos caminos independientes al mismo punto. **El nivel del panel held-out no es
  interpretable**, solo la forma.
- **Figura 3** muestra que el peor modelo por val loss no genera visiblemente peor. Con 4 muestras eso es todo lo que se puede
  afirmar: **nada** sobre diversidad ni calidad promedio.
- **Figura 4** contesta la pregunta de las copias con la única lectura válida: el **corrimiento respecto del nulo**. El nulo no
  es opcional en un dataset de una sola clase con caras alineadas.
- **La unidad del trabajo**: la columna `max_i w_i` de la Figura 2 muestra en qué deciles $\hat x_0^{\text{emp}}$ **ya es** el
  vecino más cercano. La Figura 4 es esa misma medición en el límite $t\to0$, no un apéndice.

### Límites que conviene declarar

- Los checkpoints de 15k/30k vienen de la corrida de 60k y el de 120k de la larga, porque `keep_last_checkpoints: 4` borró los
  snapshots viejos de la segunda. Las configs son idénticas salvo `num_steps`, y el notebook 08 cuantifica el piso de
  no-determinismo en el solapamiento de 60k: **ninguna diferencia por debajo de ese piso es interpretable**.
- $E(t)$ se estima con `N_EVAL_E` puntos por decil; el ruido de Monte Carlo del cociente no está acotado acá. Para el número
  que vaya al texto conviene repetir con otra `SEED_EVAL_E` y reportar la dispersión.
- La distancia $\sqrt{1-\rho}$ es invariante a brillo y contraste globales **por diseño**, pero no a traslaciones,
  rotaciones ni cambios de escala: una copia desplazada dos píxeles no la detecta. Es una cota inferior de la memorización,
  no una medida completa.